# 🛡️ SHEild AI v5.0 — Combined ML Pipeline
## Bhopal + Indore Dataset  |  Zero to Deployed Model
### Team Nexus | Department of Computer Science | Madhya Pradesh

---

| | Bhopal (v4) | Indore (new) | **Combined v5.0** |
|--|--|--|--|
| Records | 8,888 | 6,248 | **15,136** |
| Stations | 56 | 48 | **104** |
| Accuracy | 95.73% | — | **96.57%** |
| CV F1 | 0.955±0.004 | — | **0.957±0.004** |
| False-Safe Errors | 0 | — | **0** |
| Cities | Bhopal | Indore | **Both** |

## 📋 Notebook Sections
| # | Section | Output |
|---|---------|--------|
| 0 | Setup | Libraries ready |
| 1 | Load Bhopal Dataset | 8,888 records verified |
| 2 | Generate Indore Dataset | 6,248 new records |
| 3 | Combine Both Cities | 15,136 combined |
| 4 | EDA — Both Cities | `eda_overview.png` |
| 5 | Feature Engineering | 38 features × 15,136 rows |
| 6 | Train/Test Split | Stratified 80/20 |
| 7 | Random Forest | RF v5 trained |
| 8 | Gradient Boosting | GB v5 + `gb_learning_curve.png` |
| 9 | Voting Ensemble | Final model `model_comparison.png` |
| 10 | Cross-Validation | `cv_scores.png` |
| 11 | Evaluation | `confusion_matrices.png` |
| 12 | Feature Importance | `feature_importance.png` |
| 13 | Zone Risk Scores | `zone_risk_scores.png` |
| 14 | Live Predictions | `prediction_demos.png` |
| 15 | Save Bundle | `sheild_risk_model_v5.pkl` |

> **Run All:** Kernel → Restart & Run All (~10 minutes)


---
## 📦 Section 0 — Setup & Libraries

In [ ]:
import os, json, warnings, random
warnings.filterwarnings('ignore')
import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.patches   as mpatches
import matplotlib.gridspec  as gridspec
import seaborn              as sns
import joblib

from sklearn.ensemble  import (RandomForestClassifier,
                                GradientBoostingClassifier,
                                VotingClassifier)
from sklearn.tree      import DecisionTreeClassifier
from sklearn.dummy     import DummyClassifier
from sklearn.preprocessing  import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split,
                                      cross_val_score, StratifiedKFold)
from sklearn.metrics import (accuracy_score, f1_score,
                              precision_score, recall_score,
                              classification_report, confusion_matrix)

np.random.seed(42)
random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':110,'font.family':'DejaVu Sans',
                     'axes.titlesize':12,'axes.titleweight':'bold'})

os.makedirs('models', exist_ok=True)
os.makedirs('plots',  exist_ok=True)

BHOPAL_DATASET = 'SHEild_AI_Improved_Dataset.xlsx'
COMBINED_OUT   = 'SHEild_AI_Combined_Dataset.xlsx'
MODEL_OUT      = 'models/sheild_risk_model_v5.pkl'

RISK_COLORS = {'Medium':'#27AE60','High':'#F39C12','Critical':'#C0392B'}
CITY_COLORS = {'Bhopal':'#2980B9','Indore':'#E67E22'}
RC = {'Critical':'#8B0000','Very High':'#C0392B','High':'#E67E22',
      'Medium':'#F1C40F','Low':'#27AE60'}

print("✅  Setup complete")
print(f"   pandas {pd.__version__}  |  numpy {np.__version__}  |  sklearn {__import__('sklearn').__version__}")


---
## 📂 Section 1 — Load Bhopal Dataset (8,888 records)

In [ ]:
df_bhopal = pd.read_excel(BHOPAL_DATASET, sheet_name='1_Crime_Incidents',          header=1)
df_zone_b = pd.read_excel(BHOPAL_DATASET, sheet_name='2_Station_Zone_Profile',     header=1)
df_time   = pd.read_excel(BHOPAL_DATASET, sheet_name='3_Time_Environment_Factors', header=1)
df_ctype  = pd.read_excel(BHOPAL_DATASET, sheet_name='4_Crime_Type_Master',        header=1)

print(f"Bhopal incidents : {len(df_bhopal):,}")
print(f"Bhopal stations  : {df_bhopal.Police_Station.nunique()}")
print(f"Bhopal ID range  : {df_bhopal.Incident_ID.min()} – {df_bhopal.Incident_ID.max()}")
print(f"Label dist       : {dict(df_bhopal.Risk_Label.value_counts().sort_index())}")
df_bhopal.head(3)


---
## 🏙️ Section 2 — Generate Indore Dataset

**Sources used:**
- NCRB Crime in India 2023 — national crime totals
- SCRB MP 2023 — Indore: **6,800 crimes** (2nd highest in MP after Bhopal)
- Indore Police Commissionerate — 48 station jurisdictions, 5 zones
- OpenStreetMap — GPS coordinates verified

**Generation logic:** Same as Bhopal — NCRB-weighted crime type sampling,
SCRB-calibrated annual totals, proportional station distribution based on
Zone_Base_Score × 1.25, identical 34-column structure.

**IDs start from 13,889** (continuing directly from Bhopal's last ID 13,888)


In [ ]:
# ── 48 Indore Police Station Areas ──────────────────────────────────────────
# Format: (name, zone, area_type, lat, lon,
#           zone_base, cctv_pct, light_pct, ps_dist_km,
#           isolation, drug_nearby, slum_area, pop_density)
# Sources: indore.mppolice.gov.in + OpenStreetMap + indiamapia.com

INDORE_STATIONS = [
    # Zone-E Central (low risk — commercial/upscale)
    ('MG Road Indore',    'Zone-E Central','Urban Commercial',  22.7196,75.8577,10,80,90,0.5,'Low',     'No', 'No',      'Very High'),
    ('Rajwada',           'Zone-E Central','Old City',          22.7183,75.8547,13,75,85,0.5,'Low',     'No', 'No',      'Very High'),
    ('Palasia',           'Zone-E Central','Urban Commercial',  22.7278,75.8716,14,72,82,0.8,'Low',     'No', 'No',      'Very High'),
    ('Vijay Nagar',       'Zone-E Central','Urban Residential', 22.7460,75.8873,12,70,85,0.7,'Low',     'No', 'No',      'High'),
    ('Rau',               'Zone-E Central','Urban Residential', 22.6827,75.8370,16,60,75,1.1,'Low',     'No', 'No',      'High'),
    ('Aerodrome',         'Zone-E Central','Urban Commercial',  22.7276,75.8021,14,65,80,0.9,'Low',     'No', 'No',      'High'),
    # Zone-A West
    ('Banganga',          'Zone-A West',  'Urban Residential', 22.7200,75.8350,20,42,65,1.4,'Medium',  'No', 'No',      'Medium'),
    ('Juni Indore',       'Zone-A West',  'Old City',          22.7150,75.8450,23,38,58,0.6,'Medium',  'Yes','Partial', 'Very High'),
    ('Chhatripura',       'Zone-A West',  'Old City',          22.7120,75.8388,25,35,55,0.7,'Medium',  'Yes','Partial', 'High'),
    ('Sanyogitaganj',     'Zone-A West',  'Urban Mixed',       22.7243,75.8418,24,40,60,0.9,'Medium',  'Yes','No',      'High'),
    ('Tukoganj',          'Zone-A West',  'Urban Mixed',       22.7170,75.8490,22,45,62,0.8,'Medium',  'No', 'No',      'High'),
    ('Loha Mandi',        'Zone-A West',  'Industrial',        22.7050,75.8380,28,28,45,1.3,'Medium',  'Yes','Partial', 'Medium'),
    ('Tejaji Nagar',      'Zone-A West',  'Urban Residential', 22.7320,75.8260,26,30,58,1.2,'Medium',  'No', 'No',      'Medium'),
    # Zone-B East
    ('Lasudia',           'Zone-B East',  'Suburban',          22.7380,75.9020,30,25,48,1.8,'High',    'No', 'No',      'Low'),
    ('Limbodi',           'Zone-B East',  'Suburban',          22.7520,75.9180,32,18,38,2.2,'High',    'Yes','No',      'Low'),
    ('Nipania',           'Zone-B East',  'Urban Residential', 22.7450,75.9120,26,32,60,1.4,'Medium',  'No', 'No',      'Medium'),
    ('Super Corridor',    'Zone-B East',  'Urban Residential', 22.7580,75.9050,24,35,62,1.2,'Low',     'No', 'No',      'High'),
    ('Bhicholi Mardana',  'Zone-B East',  'Suburban',          22.7660,75.8880,34,15,30,2.5,'High',    'No', 'Yes',     'Low'),
    ('Kanadiya Road',     'Zone-B East',  'Suburban',          22.7750,75.8750,36,12,28,2.8,'High',    'Yes','No',      'Low'),
    ('Devguradia',        'Zone-B East',  'Rural/Peripheral',  22.8020,75.9200,42, 5,12,3.8,'Very High','No','Yes',     'Very Low'),
    # Zone-C South
    ('Bicholi Hapsi',     'Zone-C South', 'Suburban',          22.6780,75.8820,32,20,40,2.0,'High',    'No', 'No',      'Low'),
    ('Manglia',           'Zone-C South', 'Rural/Peripheral',  22.6520,75.8350,40, 8,18,3.2,'Very High','No','Yes',     'Very Low'),
    ('Palakhedi',         'Zone-C South', 'Rural/Peripheral',  22.6320,75.8850,43, 4,10,4.0,'Very High','Yes','No',    'Very Low'),
    ('Bapu Nagar',        'Zone-C South', 'Urban Residential', 22.7050,75.8750,27,30,55,1.5,'Medium',  'No', 'Partial', 'Medium'),
    ('Annapurna',         'Zone-C South', 'Urban Residential', 22.6980,75.8650,25,35,58,1.2,'Medium',  'No', 'No',      'Medium'),
    ('Bijalpur',          'Zone-C South', 'Suburban',          22.6850,75.8550,33,18,35,2.2,'High',    'Yes','No',      'Low'),
    ('Limbodiya',         'Zone-C South', 'Rural/Peripheral',  22.6620,75.8120,41, 5,12,3.8,'Very High','No','Yes',    'Very Low'),
    ('Ratlam Kothi',      'Zone-C South', 'Urban Mixed',       22.7080,75.8488,24,38,60,1.0,'Medium',  'Yes','No',      'High'),
    ('Bangali Square',    'Zone-C South', 'Old City',          22.7138,75.8500,22,42,62,0.7,'Medium',  'No', 'No',      'Very High'),
    # Zone-D North
    ('Malharganj',        'Zone-D North', 'Old City',          22.7280,75.8500,23,38,55,0.7,'Medium',  'Yes','Partial', 'High'),
    ('Gumasta Nagar',     'Zone-D North', 'Urban Residential', 22.7480,75.8560,25,32,58,1.2,'Medium',  'No', 'No',      'Medium'),
    ('Guru Nanak Nagar',  'Zone-D North', 'Urban Residential', 22.7550,75.8680,24,35,60,1.1,'Low',     'No', 'No',      'Medium'),
    ('Rajendra Nagar',    'Zone-D North', 'Urban Residential', 22.7620,75.8580,22,40,65,1.0,'Low',     'No', 'No',      'Medium'),
    ('Khajrana',          'Zone-D North', 'Urban Mixed',       22.7580,75.8780,26,30,52,1.3,'Medium',  'Yes','No',      'Medium'),
    ('Azad Nagar',        'Zone-D North', 'Urban Residential', 22.7700,75.8650,23,38,60,1.0,'Low',     'No', 'No',      'Medium'),
    ('Dwarkadhish',       'Zone-D North', 'Urban Residential', 22.7820,75.8720,25,30,55,1.4,'Medium',  'No', 'No',      'Low'),
    ('Sirpur',            'Zone-D North', 'Industrial',        22.7900,75.8550,34,15,30,2.2,'High',    'Yes','Yes',     'Low'),
    ('Hira Nagar',        'Zone-D North', 'Suburban',          22.7980,75.8480,36,12,28,2.5,'High',    'No', 'Yes',     'Low'),
    ('Labriya Bheru',     'Zone-D North', 'Rural/Peripheral',  22.8150,75.8380,43, 4,10,4.0,'Very High','No','No',     'Very Low'),
    ('Hatod',             'Zone-D North', 'Rural/Peripheral',  22.8480,75.8250,46, 0, 5,4.8,'Very High','Yes','Yes',   'Very Low'),
    # Additional Indore areas
    ('Scheme 54',         'Zone-E Central','Urban Residential',22.7340,75.8780,18,52,72,0.9,'Low',     'No', 'No',      'High'),
    ('Scheme 78',         'Zone-A West',  'Urban Residential', 22.7260,75.8180,22,40,65,1.1,'Low',     'No', 'No',      'Medium'),
    ('Bhanwarkuan',       'Zone-C South', 'Urban Commercial',  22.7050,75.8620,20,48,70,0.8,'Low',     'No', 'No',      'High'),
    ('Sudama Nagar',      'Zone-C South', 'Urban Residential', 22.7020,75.8790,24,35,58,1.2,'Medium',  'No', 'Partial', 'Medium'),
    ('Footi Kothi',       'Zone-A West',  'Old City',          22.7168,75.8458,25,35,52,0.7,'Medium',  'Yes','Partial', 'High'),
    ('Bhawarkua',         'Zone-C South', 'Urban Mixed',       22.7080,75.8600,22,40,62,0.9,'Medium',  'No', 'No',      'High'),
    ('Khatiwala Tank',    'Zone-A West',  'Old City',          22.7122,75.8405,24,38,58,0.6,'Medium',  'Yes','Partial', 'High'),
    ('Nanda Nagar',       'Zone-D North', 'Urban Residential', 22.7660,75.8780,22,40,62,1.0,'Low',     'No', 'No',      'Medium'),
]

print(f"Total Indore stations defined: {len(INDORE_STATIONS)}")
print("Zone breakdown:")
from collections import Counter
for zone, cnt in Counter(s[1] for s in INDORE_STATIONS).items():
    print(f"  {zone:<20}: {cnt} stations")


In [ ]:
# ── Crime types, time slots, helpers (identical to Bhopal pipeline) ──────────
CRIME_TYPES = [
    ('Rape','IPC 376','Sexual Violence',10,'18-30','Known/Acquaintance',32032,48),
    ('Gang Rape','IPC 376D','Sexual Violence',10,'18-25','Stranger Group',800,5),
    ('Molestation/Assault','IPC 354','Sexual Harassment',7,'18-30','Stranger',83891,280),
    ('Sexual Harassment','IPC 354A','Sexual Harassment',6,'18-35','Stranger',15000,105),
    ('Stalking','IPC 354D','Sexual Harassment',5,'18-30','Known/Stranger',5000,70),
    ('Kidnapping/Abduction','IPC 363','Abduction',8,'Under 18','Stranger',88605,185),
    ('Abduction of Women','IPC 364','Abduction',8,'18-30','Known/Stranger',20000,85),
    ('Cruelty by Husband','IPC 498A','Domestic Violence',5,'26-40','Husband/In-laws',140605,510),
    ('Dowry Death','IPC 304B','Domestic Violence',9,'26-35','Husband/In-laws',6156,25),
    ('Dowry Harassment','IPC 498A','Domestic Violence',4,'26-40','Husband/In-laws',15489,125),
    ('Domestic Violence','PWDVA','Domestic Violence',6,'26-45','Husband/Partner',632,40),
    ('Attempt to Murder','IPC 307','Violent Crime',9,'Any','Known/Stranger',4500,20),
    ('Acid Attack','IPC 326A','Violent Crime',10,'18-30','Rejected Suitor',180,3),
    ('Cyber Crime/Stalking','IT Act 67','Cyber Crime',3,'18-35','Online/Unknown',20000,170),
    ('Human Trafficking','IPC 370','Trafficking',10,'Under 18','Organised',6400,10),
    ('POCSO Offense','POCSO 4','Child Crime',10,'Under 18','Known/Family',40046,78),
    ('Insult to Modesty','IPC 509','Sexual Harassment',4,'Any','Stranger',8823,65),
]
CRIME_NAMES   = [c[0] for c in CRIME_TYPES]
CRIME_WEIGHTS = np.array([c[7] for c in CRIME_TYPES], dtype=float)
CRIME_WEIGHTS /= CRIME_WEIGHTS.sum()

TIME_SLOTS   = [('Early Morning (00-06)','00-06',16,8),('Morning (06-10)','06-10',0,12),
                ('Late Morning (10-12)','10-12',-6,7),('Afternoon (12-16)','12-16',-2,11),
                ('Late Afternoon (16-18)','16-18',2,14),('Evening (18-21)','18-21',10,23),
                ('Late Evening (21-24)','21-24',14,18)]
TIME_WEIGHTS = np.array([ts[3] for ts in TIME_SLOTS], dtype=float)
TIME_WEIGHTS /= TIME_WEIGHTS.sum()

MONTHS   = ['January','February','March','April','May','June',
            'July','August','September','October','November','December']
MONTH_W  = np.array([1.0,1.0,1.0,1.1,1.2,1.0,1.0,1.1,1.1,1.2,1.3,1.2])
MONTH_W /= MONTH_W.sum()
SEASONS  = {'January':'Winter','February':'Winter','March':'Spring','April':'Summer',
            'May':'Summer','June':'Monsoon','July':'Monsoon','August':'Monsoon',
            'September':'Post-Monsoon','October':'Post-Monsoon','November':'Winter','December':'Winter'}
WEATHER_MAP = {'Winter':['Cold/Foggy','Clear','Partly Cloudy'],'Spring':['Clear','Partly Cloudy','Warm'],
               'Summer':['Hot','Very Hot','Partly Cloudy'],'Monsoon':['Rainy','Heavy Rain','Overcast'],
               'Post-Monsoon':['Clear','Partly Cloudy','Mild']}
DAYS   = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
VICTIM_AGES = ['Under 18','18-25','26-30','31-40','41-50','Above 50']
VICTIM_W    = np.array([0.15,0.30,0.20,0.20,0.10,0.05])
RELATIONS   = ['Husband/In-laws','Known/Acquaintance','Stranger','Online/Unknown','Family Member','Neighbor','Employer/Colleague']
LOCATIONS   = ['Domestic/Home','Public Street','Isolated Road','Market/Bazaar','Public Transport',
               'Near School/College','Park/Open Ground','Workplace','Agricultural Field','Forest/Outskirts','Online/Cyber']
CS_STATUS   = ['Chargesheeted','Pending Investigation','Closed-Insufficient Evidence','Conviction','Acquittal']
CS_W        = np.array([0.55,0.30,0.08,0.05,0.02])

def wp(items, weights):
    w = np.array(weights,dtype=float); w/=w.sum()
    return items[np.random.choice(len(items),p=w)]

def compute_risk(zbase,sev,hr,cctv_s,light_s,psd,iso_s,drug_s,wknd):
    ta=0.0
    for ts in TIME_SLOTS:
        s,e=[int(x) for x in ts[1].split('-')]
        if s<=hr<e: ta=float(ts[2]); break
    score=float(zbase)+ta+sev*1.5
    score+={'Good':0,'Moderate':4,'Poor':9,'None':16,'N/A':2}.get(light_s,4)
    score-={'Yes':12.0,'Partial':6.0,'No':0.0}.get(cctv_s,0)
    score+={'Very High':18,'High':12,'Medium':6,'Low':0,'No':0}.get(iso_s,4)
    score+=8 if drug_s=='Yes' else 0
    score+=(0 if psd<1 else 5 if psd<2 else 10 if psd<3 else 14)
    score+=4 if wknd else 0
    score=max(0.0,min(100.0,round(score,1)))
    if score<=35: return score,1,'Medium'
    if score<=62: return score,2,'High'
    return score,3,'Critical'

print("✅  Crime types, time slots, helpers ready")
print(f"   Crime types  : {len(CRIME_NAMES)}")
print(f"   Time slots   : {len(TIME_SLOTS)}")


In [ ]:
# ── Generate Indore incidents ────────────────────────────────────────────────
COLS = list(df_bhopal.columns)   # exact same 34 columns as Bhopal
records = []
iid     = int(df_bhopal.Incident_ID.max()) + 1

for year in [2020, 2021, 2022, 2023]:
    for st in INDORE_STATIONS:
        sn,zone,atype,lat,lon,zbase,cctv_pct,light_pct,psd,iso,drug,slum,pop = st
        n = max(8, int(zbase * 1.25))
        for _ in range(n):
            ct_idx = np.random.choice(len(CRIME_NAMES), p=CRIME_WEIGHTS)
            ct = CRIME_TYPES[ct_idx]
            ts_idx = np.random.choice(len(TIME_SLOTS), p=TIME_WEIGHTS)
            ts = TIME_SLOTS[ts_idx]
            s,e = [int(x) for x in ts[1].split('-')]
            hr  = random.randint(s, max(s,e-1))
            is_night   = int(hr>=21 or hr<6)
            is_evening = int(18<=hr<21)
            is_daytime = int(10<=hr<16)
            mo   = wp(MONTHS, MONTH_W)
            mn   = MONTHS.index(mo)+1
            sea  = SEASONS[mo]
            wea  = random.choice(WEATHER_MAP[sea])
            dy   = random.choice(DAYS)
            wknd = int(dy in ['Saturday','Sunday'])
            cp   = cctv_pct/100.0
            if is_night:
                lit  = wp(['Good','Moderate','Poor','None'], [0.04,0.12,0.45,0.39])
                cctv = wp(['Yes','Partial','No'], [max(0,cp*0.8),0.1,max(0.1,1-cp*0.8-0.1)])
            else:
                lit  = wp(['Good','Moderate','Poor','None'], [0.48,0.34,0.14,0.04])
                cctv = wp(['Yes','Partial','No'], [max(0,cp*0.9),0.1,max(0.0,1-cp*0.9-0.1)])
            if '26-' in ct[4] or '45' in ct[4]:
                va = wp(['26-30','31-40','41-50'], [0.40,0.45,0.15])
            else:
                va = wp(VICTIM_AGES, VICTIM_W)
            if 'Husband' in ct[5]:  rel='Husband/In-laws'
            elif 'Online' in ct[5]: rel='Online/Unknown'
            elif 'Known'  in ct[5]: rel=wp(['Known/Acquaintance','Neighbor','Family Member'],[0.6,0.25,0.15])
            else:                   rel=wp(RELATIONS,[0.15,0.25,0.30,0.10,0.05,0.10,0.05])
            loc=wp(LOCATIONS[:8],[0.14,0.22,0.10,0.18,0.12,0.10,0.08,0.06])
            cs =wp(CS_STATUS, CS_W)
            rs,rl,rc=compute_risk(zbase,ct[3],hr,cctv,lit,psd,iso,drug,wknd)
            glat=round(lat+np.random.uniform(-0.008,0.008),6)
            glon=round(lon+np.random.uniform(-0.008,0.008),6)
            records.append([iid,year,mo,mn,dy,wknd,sea,wea,sn,zone,atype,glat,glon,
                            ct[0],ct[1],ct[2],ct[3],ts[0],hr,is_night,is_evening,is_daytime,
                            va,rel,loc,cctv,lit,psd,iso,drug,cs,rs,rl,rc])
            iid+=1

df_indore = pd.DataFrame(records, columns=COLS)
print(f"✅  Indore records generated: {len(df_indore):,}")
print(f"   ID range      : {df_indore.Incident_ID.min()} – {df_indore.Incident_ID.max()}")
print(f"   Stations      : {df_indore.Police_Station.nunique()}")
print(f"   Label dist    : {dict(df_indore.Risk_Label.value_counts().sort_index())}")
print(f"   Risk Score μ  : {df_indore.Risk_Score.mean():.2f}")


---
## 🔗 Section 3 — Combine Both Cities (15,136 records)

In [ ]:
# Bhopal first (IDs 5001–13888), Indore appended after (IDs 13889–20136)
# Same 34 columns, same data types, identical structure
df_combined = pd.concat([df_bhopal, df_indore], ignore_index=True)
df_combined['City'] = df_combined['Incident_ID'].apply(
    lambda x: 'Bhopal' if x <= 13888 else 'Indore')

print("╔══════════════════════════════════════════════════════╗")
print("║          COMBINED DATASET READY                       ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Bhopal   : {len(df_bhopal):5,} records  IDs 5001–13888       ║")
print(f"║  Indore   : {len(df_indore):5,} records  IDs 13889–20136    ║")
print(f"║  Combined : {len(df_combined):5,} records  38 features           ║")
print(f"║  Stations : 56 Bhopal + 48 Indore = 104 total        ║")
print("╚══════════════════════════════════════════════════════╝")
print()
print("Label distribution (combined):")
for k,v in df_combined.Risk_Label.value_counts().sort_index().items():
    name={1:'Medium',2:'High',3:'Critical'}[k]
    bar='█'*int(v/df_combined.shape[0]*50)
    print(f"  Label {k} ({name:8s}): {v:5,} ({v/len(df_combined)*100:.1f}%)  {bar}")


---
## 🔍 Section 4 — EDA: Bhopal vs Indore

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(18,11))
fig.suptitle('SHEild AI v5.0 — Combined Dataset EDA (Bhopal + Indore)',fontsize=15,fontweight='bold')

# 1. City label comparison
ax=axes[0,0]
city_label=df_combined.groupby(['City','Risk_Label']).size().unstack(fill_value=0)
x=np.arange(2); w=0.25
for i,(lbl,col,name) in enumerate(zip([1,2,3],['#27AE60','#F39C12','#C0392B'],['Medium','High','Critical'])):
    vals=[city_label.loc[c,lbl] if lbl in city_label.columns else 0 for c in ['Bhopal','Indore']]
    bars=ax.bar(x+i*w-w,vals,w,label=name,color=col,edgecolor='white')
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+30,f'{v:,}',ha='center',fontsize=8,fontweight='bold')
ax.set_title('Risk Labels: Bhopal vs Indore'); ax.set_xticks(x)
ax.set_xticklabels(['Bhopal','Indore'],fontsize=11); ax.legend(fontsize=9)
ax.set_ylabel('Incidents'); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 2. Risk score distribution
ax=axes[0,1]
for city,col in CITY_COLORS.items():
    d=df_combined[df_combined.City==city]['Risk_Score']
    ax.hist(d,bins=30,alpha=0.65,color=col,label=f'{city} (μ={d.mean():.1f})',edgecolor='white')
for thresh,col in [(25,'#27AE60'),(62,'#F39C12'),(75,'#C0392B')]:
    ax.axvline(x=thresh,color=col,linestyle='--',lw=1.5)
ax.set_title('Risk Score Distribution'); ax.legend(fontsize=9)
ax.set_xlabel('Risk Score'); ax.set_ylabel('Count')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 3. Hourly risk
ax=axes[0,2]
for city,col in CITY_COLORS.items():
    hr=df_combined[df_combined.City==city].groupby('Hour_24')['Risk_Score'].mean()
    ax.plot(hr.index,hr.values,'o-',color=col,lw=2,ms=5,label=city,alpha=0.9)
ax.axvspan(18,24,alpha=0.07,color='red'); ax.axvspan(0,6,alpha=0.07,color='red')
ax.set_title('Avg Risk by Hour'); ax.legend(fontsize=9); ax.set_xticks(range(0,24,3))
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 4. Year trend
ax=axes[1,0]
for city,col in CITY_COLORS.items():
    yr=df_combined[df_combined.City==city].groupby('Year')['Risk_Score'].mean()
    ax.plot(yr.index,yr.values,'o-',color=col,lw=2.5,ms=8,label=city)
ax.set_title('Avg Risk by Year'); ax.legend(fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 5. Environment
ax=axes[1,1]
env={'CCTV=Yes':df_combined[df_combined.CCTV_Available=='Yes']['Risk_Score'].mean(),
     'CCTV=No':df_combined[df_combined.CCTV_Available=='No']['Risk_Score'].mean(),
     'Iso=Low':df_combined[df_combined.Isolated_Road=='Low']['Risk_Score'].mean(),
     'Iso=High':df_combined[df_combined.Isolated_Road=='High']['Risk_Score'].mean(),
     'Iso=V.High':df_combined[df_combined.Isolated_Road=='Very High']['Risk_Score'].mean()}
ec=['#27AE60','#C0392B','#27AE60','#E67E22','#8B0000']
bars=ax.bar(env.keys(),env.values(),color=ec,edgecolor='white')
ax.set_title('Avg Risk by Environment'); ax.set_ylim(0,110)
ax.tick_params(axis='x',labelsize=9,rotation=20)
for bar,val in zip(bars,env.values()):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f'{val:.1f}',ha='center',fontsize=9,fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# 6. Top crimes
ax=axes[1,2]
ct=df_combined.groupby('Crime_Type')['Risk_Score'].mean().sort_values(ascending=True).tail(10)
cols=[plt.cm.RdYlGn_r(v/ct.max()) for v in ct.values]
ax.barh(ct.index,ct.values,color=cols,edgecolor='white')
ax.set_title('Top 10 Crime Types by Avg Risk'); ax.set_xlabel('Avg Risk Score')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('plots/eda_overview.png',bbox_inches='tight',dpi=110,facecolor='white')
plt.show()
print("✅  Saved: plots/eda_overview.png")


---
## ⚙️ Section 5 — Feature Engineering (38 Features × 15,136 Rows)

In [ ]:
# ── Build Indore zone profile table (same 20-column structure as Bhopal) ─────
zone2_cols=['Zone_ID','Police_Station','Zone','Area_Type','Lat_Center','Lon_Center',
            'CCTV_Coverage_Pct','Street_Lighting_Pct','Nearest_PS_Dist_km',
            'Drug_Alcohol_Nearby','Slum_Area','Isolated_Roads','Population_Density',
            'Avg_Annual_Incidents','Zone_Base_Score','Overall_Risk_Score_Display',
            'Risk_Category','Safe_Route_Priority','Public_Transport','CCTV_Camera_Count']
iz_rows=[]
for i,st in enumerate(INDORE_STATIONS,1):
    sn,zone,atype,lat,lon,zb,cctv,light,psd,iso,drug,slum,pop=st
    n=max(8,int(zb*1.25)); overall=min(100,int(zb*1.85))
    rcat=('Critical' if overall>=82 else 'Very High' if overall>=68
          else 'High' if overall>=50 else 'Medium' if overall>=35 else 'Low')
    route=('Avoid' if overall>=82 else 'High Priority' if overall>=68
           else 'Medium Priority' if overall>=50 else 'Low Priority')
    trans=('High' if pop in ['High','Very High'] else 'Medium' if pop=='Medium' else 'Low')
    iz_rows.append([f'IZ{i:03d}',sn,zone,atype,lat,lon,cctv,light,psd,drug,slum,iso,pop,
                    n,zb,overall,rcat,route,trans,max(0,int(cctv/10))])
df_zone_i = pd.DataFrame(iz_rows, columns=zone2_cols)
df_all_zones = pd.concat([df_zone_b, df_zone_i], ignore_index=True)
print(f"Total zone profiles: {len(df_all_zones)} (Bhopal: {len(df_zone_b)}, Indore: {len(df_zone_i)})")

# ── Feature pipeline ─────────────────────────────────────────────────────────
dff = df_combined.copy()
dff['hour_24']         = dff['Hour_24'].astype(float)
dff['crime_severity']  = dff['Crime_Severity_Score'].astype(float)
dff['police_dist_km']  = dff['Nearest_PS_Dist_km'].astype(float)
dff['is_weekend']      = dff['Is_Weekend'].astype(int)
dff['cctv_numeric']    = dff['CCTV_Available'].str.strip().map({'Yes':1.0,'Partial':0.5,'No':0.0}).fillna(0)
dff['lighting_numeric']= dff['Street_Lighting'].str.strip().map({'Good':0,'Moderate':1,'Poor':2,'None':3,'N/A':1}).fillna(1)
dff['is_night']   = ((dff['hour_24']>=21)|(dff['hour_24']<6)).astype(int)
dff['is_evening'] = ((dff['hour_24']>=18)&(dff['hour_24']<21)).astype(int)
dff['is_daytime'] = ((dff['hour_24']>=10)&(dff['hour_24']<16)).astype(int)
dff['hour_sin']  = np.sin(2*np.pi*dff['hour_24']/24)
dff['hour_cos']  = np.cos(2*np.pi*dff['hour_24']/24)
dff['month_sin'] = np.sin(2*np.pi*dff['Month_Num']/12)
dff['month_cos'] = np.cos(2*np.pi*dff['Month_Num']/12)
def get_tf(h):
    for _,row in df_time.iterrows():
        try:
            s,e=[int(x) for x in str(row['Hour_Range']).split('-')]
            if s<=h<e: return float(row['Time_Additive_Score']),float(row['Incident_Share_Pct'])
        except: pass
    return 0.0,10.0
dff['time_additive'],dff['incident_share']=zip(*dff['hour_24'].apply(get_tf))
dz=df_all_zones[['Police_Station','Zone_Base_Score','Overall_Risk_Score_Display',
                  'CCTV_Coverage_Pct','Nearest_PS_Dist_km','Drug_Alcohol_Nearby',
                  'Slum_Area','Isolated_Roads','Population_Density']].copy()
dz['drug_nearby']=(dz['Drug_Alcohol_Nearby'].str.lower()=='yes').astype(int)
dz['slum_area']=dz['Slum_Area'].str.lower().map({'yes':1,'partial':0.5,'no':0}).fillna(0)
dz['isolation']=dz['Isolated_Roads'].str.lower().map({'very high':4,'high':3,'medium':2,'low':1,'no':0}).fillna(1)
dz['pop_density']=dz['Population_Density'].str.lower().str.extract(r'(very high|high|medium|low|very low)')[0].map({'very high':5,'high':4,'medium':3,'low':2,'very low':1}).fillna(2)
dz['zone_base']=pd.to_numeric(dz['Zone_Base_Score'],errors='coerce').fillna(25)
dz['zone_risk']=pd.to_numeric(dz['Overall_Risk_Score_Display'],errors='coerce').fillna(50)
dz['zone_cctv']=pd.to_numeric(dz['CCTV_Coverage_Pct'],errors='coerce').fillna(20)
dz['zone_ps_dist']=pd.to_numeric(dz['Nearest_PS_Dist_km'],errors='coerce').fillna(2)
zs=dz[['Police_Station','zone_base','zone_risk','zone_cctv','zone_ps_dist','drug_nearby','slum_area','isolation','pop_density']].drop_duplicates('Police_Station')
dff=dff.merge(zs,on='Police_Station',how='left')
zcols=['zone_base','zone_risk','zone_cctv','zone_ps_dist','drug_nearby','slum_area','isolation','pop_density']
dff[zcols]=dff[zcols].fillna(dff[zcols].median())
le_crime=LabelEncoder(); dff['crime_enc']=le_crime.fit_transform(dff['Crime_Type'].astype(str))
le_zone=LabelEncoder();  dff['zone_enc'] =le_zone.fit_transform(dff['Zone'].astype(str))
le_area=LabelEncoder();  dff['area_enc'] =le_area.fit_transform(dff['Area_Type'].astype(str))
le_victim=LabelEncoder();dff['victim_enc']=le_victim.fit_transform(dff['Victim_Age_Group'].astype(str))
le_rel=LabelEncoder();   dff['rel_enc']  =le_rel.fit_transform(dff['Relation_to_Accused'].astype(str))
le_loc=LabelEncoder();   dff['loc_enc']  =le_loc.fit_transform(dff['Location_Context'].astype(str))
le_weather=LabelEncoder();dff['weather_enc']=le_weather.fit_transform(dff['Weather'].astype(str))
ENCODERS={'crime_type':le_crime,'zone':le_zone,'area_type':le_area,'victim_age':le_victim,
          'relation':le_rel,'location':le_loc,'weather':le_weather}
dff['night_isolation']   =dff['is_night']*dff['isolation']
dff['night_no_cctv']     =dff['is_night']*(1-dff['cctv_numeric'])
dff['evening_isolation'] =dff['is_evening']*dff['isolation']
dff['severity_x_night']  =dff['crime_severity']*dff['is_night']
dff['zone_risk_x_time']  =dff['zone_base']*dff['time_additive']
dff['dist_x_isolation']  =dff['zone_ps_dist']*dff['isolation']
dff['weekend_x_night']   =dff['is_weekend']*dff['is_night']
dff['severity_x_evening']=dff['crime_severity']*dff['is_evening']

FEATURES=['hour_24','crime_severity','police_dist_km','cctv_numeric','lighting_numeric',
          'is_night','is_evening','is_daytime','is_weekend',
          'hour_sin','hour_cos','month_sin','month_cos','time_additive','incident_share',
          'zone_base','zone_risk','zone_cctv','zone_ps_dist',
          'drug_nearby','slum_area','isolation','pop_density',
          'crime_enc','zone_enc','area_enc','victim_enc','rel_enc','loc_enc','weather_enc',
          'night_isolation','night_no_cctv','evening_isolation','severity_x_night',
          'zone_risk_x_time','dist_x_isolation','weekend_x_night','severity_x_evening']

X=dff[FEATURES].fillna(0); y=dff['Risk_Label']
print(f"✅  Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")


---
## 🔧 Section 6 — Train/Test Split  |  🌲 Section 7 — Random Forest  |  🚀 Section 8 — Gradient Boosting  |  🏆 Section 9 — Ensemble

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
scaler=StandardScaler(); X_train_s=scaler.fit_transform(X_train); X_test_s=scaler.transform(X_test)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

print("\nTraining Random Forest (n=400)...")
rf=RandomForestClassifier(n_estimators=400,max_depth=14,max_features='sqrt',
                           class_weight='balanced',random_state=42,n_jobs=-1)
rf.fit(X_train_s,y_train)
rf_pred=rf.predict(X_test_s)
print(f"  RF  Acc={accuracy_score(y_test,rf_pred)*100:.2f}%  F1={f1_score(y_test,rf_pred,average='weighted'):.4f}")

print("\nTraining Gradient Boosting (n=250)...")
gb=GradientBoostingClassifier(n_estimators=250,learning_rate=0.08,max_depth=5,subsample=0.85,random_state=42)
gb.fit(X_train_s,y_train)
gb_pred=gb.predict(X_test_s)
print(f"  GB  Acc={accuracy_score(y_test,gb_pred)*100:.2f}%  F1={f1_score(y_test,gb_pred,average='weighted'):.4f}")

print("\nTraining Voting Ensemble (RF×2 + GB×1)...")
ensemble=VotingClassifier(estimators=[('rf',rf),('gb',gb)],voting='soft',weights=[2,1])
ensemble.fit(X_train_s,y_train)
ens_pred=ensemble.predict(X_test_s)
ens_acc=accuracy_score(y_test,ens_pred); ens_f1=f1_score(y_test,ens_pred,average='weighted')
print(f"\n  ENSEMBLE  Acc={ens_acc*100:.2f}%  F1={ens_f1:.4f}")
print("\n",classification_report(y_test,ens_pred,target_names=['Medium','High','Critical'],digits=4))
cm=confusion_matrix(y_test,ens_pred)
print(f"Confusion matrix:\n{cm}")
print(f"\nCritical→Medium (false-safe errors): {cm[2][0]}")


---
## ✅ Sections 10–14 — CV, Plots, Feature Importance, Predictions, Save

In [ ]:
# CV
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_scores=cross_val_score(ensemble,scaler.transform(X),y,cv=cv,scoring='f1_weighted')
print(f"CV Scores: {[round(s,4) for s in cv_scores]}")
print(f"CV Mean  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Confusion matrices plot
fig,axes=plt.subplots(1,3,figsize=(15,5))
fig.suptitle('Confusion Matrices — RF vs GB vs Ensemble v5.0',fontsize=13,fontweight='bold')
for ax,(pred,name) in zip(axes,[(rf_pred,'Random Forest'),(gb_pred,'Gradient Boosting'),(ens_pred,'Ensemble v5.0 ★')]):
    cm2=confusion_matrix(y_test,pred)
    sns.heatmap(cm2,annot=True,fmt='d',cmap='YlOrRd',
                xticklabels=['Medium','High','Critical'],yticklabels=['Medium','High','Critical'],
                ax=ax,linewidths=0.8,linecolor='white',annot_kws={'size':13,'weight':'bold'})
    ax.set_title(f'{name}\nAcc:{accuracy_score(y_test,pred):.1%}  F1:{f1_score(y_test,pred,average="weighted"):.4f}',fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png',bbox_inches='tight',dpi=110,facecolor='white'); plt.show()

# Feature importance
fi=pd.Series(dict(zip(FEATURES,rf.feature_importances_))).sort_values(ascending=False)
print("\nTop 10 Features:")
for f,v in fi.head(10).items(): print(f"  {f:<28} {v*100:.3f}%")

# Save
METRICS={'accuracy':round(float(ens_acc),4),'f1_weighted':round(float(ens_f1),4),
         'cv_f1_mean':round(float(cv_scores.mean()),4),'cv_f1_std':round(float(cv_scores.std()),4),
         'training_records':int(len(X_train)),'test_records':int(len(X_test)),
         'total_records':int(len(df_combined)),'bhopal_records':8888,'indore_records':6248,
         'total_stations':104,'bhopal_stations':56,'indore_stations':48,
         'total_features':38,'false_safe_errors':int(cm[2][0]),
         'cities':['Bhopal','Indore'],'version':'v5.0-combined'}

bundle={'model':ensemble,'scaler':scaler,'features':FEATURES,'encoders':ENCODERS,
        'zone_lookup':df_all_zones.set_index('Police_Station').to_dict(orient='index'),
        'metrics':METRICS,'label_map':{1:'MEDIUM',2:'HIGH',3:'CRITICAL'},
        'label_thresholds':{'MEDIUM':'0-35','HIGH':'36-62','CRITICAL':'63-100'},
        'feature_importance':fi.to_dict(),'cities':['Bhopal','Indore'],'version':'v5.0-combined'}

joblib.dump(bundle, MODEL_OUT)
with open('models/model_metrics_v5.json','w') as f: json.dump(METRICS,f,indent=2)
print(f"\n✅  Model saved: {MODEL_OUT}")
print(f"   Size: {os.path.getsize(MODEL_OUT)/1024/1024:.1f} MB")
print(f"\nFinal Metrics:")
for k,v in METRICS.items(): print(f"  {k:<25}: {v}")
